# Tiled Noise-Attack Detection on a Kaggle Dataset (2xT4)

**Goal of this notebook (the workflow you asked for):**

1. **Grab a dataset** that lives on Kaggle (animals images by default).
2. **Pull out 10 random images** from it.
3. **Run an adversarial attack on an arbitrary, non-rectangular region** (an irregular blob — not
   the whole image, not aligned to tiles) so you can *see by eye* where it was attacked.
4. **Tile** each attacked image into a **4x4 grid** with a reusable `tile_image(...)` function.
5. **Scan tile by tile** — check each tile for attack noise and **flag** the noisy ones. Because the
   attack region is an odd shape, finding *which tiles it touched* is the whole reason for tiling.
6. **Compare** side by side: the **original**, the **attacked** image with its real attack region,
   the per-tile noise score, and **true vs detected** noisy tiles.

The point: catch noisy/attacked tiles **before training** so the model is not tricked by the
perturbation. Attacks: **FGSM** (fast) or **PGD** (stronger). Runs on Kaggle with **GPU T4 x2**.

> Set the accelerator to **GPU T4 x2** and turn **Internet On** in the Kaggle notebook settings.

## 0. Install
`grad-cam` is the PyPI package for `jacobgil/pytorch-grad-cam` (imported as `pytorch_grad_cam`).

In [ ]:
!pip install -q grad-cam

## 1. Setup & GPU inventory
We list the GPUs so you can confirm both T4s are visible before running the heavy cells.

In [ ]:
# ============================================================
# CELL 1 - Setup & GPU inventory
# ============================================================
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision import models, transforms
from PIL import Image, ImageDraw
import urllib.request, json, os, time
from concurrent.futures import ThreadPoolExecutor

N_GPU = torch.cuda.device_count()
DEVICES = [f"cuda:{i}" for i in range(N_GPU)] if N_GPU else ["cpu"]
print("GPUs found:", N_GPU)
for i in range(N_GPU):
    print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")
print("Using devices:", DEVICES)

## 2. One ResNet50 per GPU
We keep an independent ResNet50 copy on each device so each GPU can work on its own slice of a
tile-batch in parallel. We also load human-readable ImageNet class names.

In [ ]:
# ============================================================
# CELL 2 - ResNet50 (ImageNet) on every GPU + labels
# ============================================================
def make_model(dev):
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    return m.eval().to(dev)

MODELS = {d: make_model(d) for d in DEVICES}          # device -> model copy
print(f"Loaded {len(MODELS)} ResNet50 copies (one per device).")

url = "https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json"
idx2label = {int(k): v[1] for k, v in json.load(urllib.request.urlopen(url)).items()}

## 3. Preprocessing + the `tile_image` function
Images live in `[0,1]` pixel space; ImageNet normalization happens *inside* the model call so the
adversarial noise stays measured in **real pixels**.

`tile_image` is the reusable tiler you asked for: it cuts a `[1,3,SIZE,SIZE]` image into a
`GRID x GRID` stack of small tiles (row-major order). `untile` is its inverse for visualization.

In [ ]:
# ============================================================
# CELL 3 - Preprocessing, the tile function, helpers
# ============================================================
SIZE = 448            # working resolution per image
GRID = 4              # GRID x GRID tiles -> 4x4 = 16 tiles per image
TILE = SIZE // GRID   # tile side in pixels (112)

to_tensor = transforms.Compose([transforms.Resize((SIZE, SIZE)), transforms.ToTensor()])

_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
def normalize(t):                       # normalize on the tensor's own device
    return (t - _mean.to(t.device)) / _std.to(t.device)

def load_image(path_or_url):
    # Load local path OR URL -> [1,3,SIZE,SIZE] float tensor in [0,1] (on CPU).
    if str(path_or_url).startswith("http"):
        fn = "/tmp/" + os.path.basename(path_or_url)
        if not os.path.exists(fn):
            urllib.request.urlretrieve(path_or_url, fn)
        path_or_url = fn
    img = Image.open(path_or_url).convert("RGB")
    return to_tensor(img).unsqueeze(0)

# ---- THE TILE FUNCTION ----
def tile_image(x, grid=GRID, tile=TILE):
    # [1,3,SIZE,SIZE] -> [grid*grid, 3, tile, tile]  (row-major tile order).
    p = x.unfold(2, tile, tile).unfold(3, tile, tile)        # [1,3,grid,grid,tile,tile]
    p = p.permute(0, 2, 3, 1, 4, 5).reshape(-1, 3, tile, tile)
    return p.contiguous()

def untile(tiles, grid=GRID, tile=TILE):
    # [grid*grid, C, tile, tile] -> [C, SIZE, SIZE] mosaic (inverse of tile_image).
    C = tiles.shape[1]
    g = tiles.reshape(grid, grid, C, tile, tile).permute(2, 0, 3, 1, 4)
    return g.reshape(C, grid * tile, grid * tile)

def tile_grid_to_full(per_tile_scalar, grid=GRID, tile=TILE):
    # [grid*grid] per-tile values -> [SIZE,SIZE] blocky heatmap for overlay.
    g = np.asarray(per_tile_scalar).reshape(grid, grid)
    return np.kron(g, np.ones((tile, tile)))

def to_np(t):  return t.squeeze().detach().cpu().permute(1, 2, 0).numpy()
def norm01(a):
    a = np.asarray(a, np.float32)
    return (a - a.min()) / (a.max() - a.min() + 1e-8)

# quick sanity check that tiling is loss-less
_t = torch.rand(1, 3, SIZE, SIZE)
assert torch.allclose(untile(tile_image(_t)), _t.squeeze(0), atol=1e-6)
print(f"tile_image OK: {GRID}x{GRID} grid, each tile {TILE}x{TILE} px, {GRID*GRID} tiles/image")

## 4. Get the dataset & pull 10 random images
Point `FOLDER` at any Kaggle image dataset. By default we look for the **Animals-10** dataset
(`alessiocorrado99/animals10`). Add it via **+ Add Input** in the Kaggle sidebar.
If no folder is found, we fall back to downloading a handful of animal samples from the web.

In [ ]:
# ============================================================
# CELL 4 - Gather 10 random images from a Kaggle dataset (or web fallback)
# ============================================================
N_IMAGES = 10

CANDIDATE_FOLDERS = [
    "/kaggle/input/animals10/raw-img",
    "/kaggle/input/imagenette/imagenette2/val",
    "/kaggle/input/imagenette2/val",
]
FOLDER = next((f for f in CANDIDATE_FOLDERS if os.path.isdir(f)), "")

def gather_images(folder, n=N_IMAGES, seed=1):
    exts = (".jpg", ".jpeg", ".png", ".JPEG")
    paths = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.endswith(exts): paths.append(os.path.join(root, f))
    rng = np.random.default_rng(seed)
    return list(rng.choice(paths, size=min(n, len(paths)), replace=False)) if paths else []

if FOLDER:
    images = gather_images(FOLDER, n=N_IMAGES)
    print(f"Picked {len(images)} random images from {FOLDER}")
else:
    base = "https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/"
    images = [base + n for n in [
        "n02099601_golden_retriever.JPEG", "n02123045_tabby.JPEG", "n02391049_zebra.JPEG",
        "n02129165_lion.JPEG", "n02129604_tiger.JPEG", "n02510455_giant_panda.JPEG",
        "n01518878_ostrich.JPEG", "n01806143_peacock.JPEG", "n01882714_koala.JPEG",
        "n02007558_flamingo.JPEG"]]
    print(f"No Kaggle folder found -- using {len(images)} downloaded animal samples")

# ---- Preview: actually SHOW the 10 images we will work on ----
ncol = 5
nrow = int(np.ceil(len(images) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3 * ncol, 3 * nrow))
for ax, p in zip(np.ravel(axes), images):
    ax.imshow(to_np(load_image(p)))             # load -> [0,1] tensor -> HxWx3
    ax.set_title(os.path.basename(str(p))[:22], fontsize=8)
    ax.axis("off")
for ax in np.ravel(axes)[len(images):]:         # hide any empty cells
    ax.axis("off")
fig.suptitle(f"The {len(images)} images we will attack & test", y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

## 5. The adversarial attack — on an arbitrary (non-rectangular) region
A real attack does **not** have to cover the whole image, and its shape does **not** line up with
our tile grid. So we confine the perturbation to an **irregular blob** (a random polygon) that
crosses tile boundaries however it likes — *that* is exactly why we then analyze the image in tile
chunks: to find which tiles the blob touched.

Two attack strengths are provided:
- **FGSM** — one signed-gradient step (fast, crude).
- **PGD** — many small projected steps inside the same `epsilon` budget (slower, much stronger; the
  standard robustness benchmark). PGD's noise is subtler, so it's a tougher test for the detector.

Both are confined to the blob mask, and both keep the image a valid `[0,1]` picture.

In [ ]:
# ============================================================
# CELL 5 - Irregular attack region + FGSM / PGD attacks
# ============================================================
def random_blob_mask(size=SIZE, seed=None, n_verts=9, rad_frac=(0.12, 0.34)):
    # Irregular, non-rectangular polygon mask -> [1,1,size,size] in {0,1}.
    rng = np.random.default_rng(seed)
    cx, cy = rng.uniform(0.30, 0.70, 2) * size
    angles = np.sort(rng.uniform(0, 2*np.pi, n_verts))
    radii = rng.uniform(*rad_frac, n_verts) * size
    pts = [(float(cx + r*np.cos(a)), float(cy + r*np.sin(a))) for a, r in zip(angles, radii)]
    im = Image.new("L", (size, size), 0)
    ImageDraw.Draw(im).polygon(pts, fill=1)
    return torch.tensor(np.array(im), dtype=torch.float32).view(1, 1, size, size)

def _grad_sign(xi, true_class, d):
    out = MODELS[d](normalize(xi))
    loss = F.cross_entropy(out, torch.tensor([true_class], device=d))
    MODELS[d].zero_grad(); loss.backward()
    return xi.grad.sign()

def fgsm_attack(x, true_class, epsilon=0.05, mask=None):
    # One signed-gradient step, confined to mask if given. Returns adv image (CPU).
    d = DEVICES[0]
    xi = x.clone().detach().to(d).requires_grad_(True)
    perturb = epsilon * _grad_sign(xi, true_class, d)
    if mask is not None: perturb = perturb * mask.to(d)
    return torch.clamp(xi.detach() + perturb, 0, 1).cpu()

def pgd_attack(x, true_class, epsilon=0.05, alpha=0.01, steps=10, mask=None):
    # Iterative PGD inside the L-inf epsilon ball, confined to mask if given. Returns adv (CPU).
    d = DEVICES[0]
    x0 = x.clone().detach().to(d)
    xi = x0.clone()
    m = mask.to(d) if mask is not None else None
    for _ in range(steps):
        xi.requires_grad_(True)
        step = alpha * _grad_sign(xi, true_class, d)
        if m is not None: step = step * m
        with torch.no_grad():
            xi = torch.min(torch.max(xi.detach() + step, x0 - epsilon), x0 + epsilon)  # project
            xi = torch.clamp(xi, 0, 1)
    return xi.detach().cpu()

def attack(x, true_class, method="pgd", epsilon=0.05, mask=None, **kw):
    if method == "fgsm":
        return fgsm_attack(x, true_class, epsilon, mask)
    return pgd_attack(x, true_class, epsilon, mask=mask, **kw)

## 6. Batched prediction across both GPUs
Used to (a) pick the FGSM target class from the clean image and (b) check whether the global label
flipped after the attack. The tile batch is split across the available GPUs and run in threads
(CUDA ops release the GIL -> real parallelism).

In [ ]:
# ============================================================
# CELL 6 - Multi-GPU batched prediction
# ============================================================
def batched_predict(tiles_cpu, use_gpus=None):
    devs = use_gpus or DEVICES
    chunks = torch.chunk(tiles_cpu, len(devs), dim=0) if len(devs) > 1 else [tiles_cpu]
    devs = devs[:len(chunks)]
    def _run(args):
        d, ch = args
        with torch.no_grad():            # no_grad is thread-local -> must re-enter it in each worker
            p = MODELS[d](normalize(ch).to(d)).softmax(1)
            conf, idx = p.max(1)
        return idx.cpu().numpy(), conf.cpu().numpy()
    with ThreadPoolExecutor(max_workers=len(devs)) as ex:
        outs = list(ex.map(_run, zip(devs, chunks)))
    idx = np.concatenate([o[0] for o in outs]); conf = np.concatenate([o[1] for o in outs])
    return idx, conf

## 7. The per-tile noise detector + ground truth
**Detector:** break the attacked image into tiles and **check each tile** for attack noise. FGSM/PGD
inject **high-frequency** energy, so we score each tile by `mean(|tile - blur(tile)|)` and flag any
tile scoring above `median + k*MAD` of a **clean** baseline (calibrated on clean tiles).

**Ground truth:** because the attack lives inside an irregular blob, a tile is "truly attacked" if
the blob covers more than `cover_thresh` of that tile's pixels. Comparing detected vs truly-attacked
tiles gives us tile-level **precision / recall / IoU** — a real localization score.

In [ ]:
# ============================================================
# CELL 7 - Per-tile noise score, calibrated detector, ground truth, metrics
# ============================================================
def _gaussian_kernel(sigma=1.0, ksize=5):
    ax = torch.arange(ksize) - ksize // 2
    g = torch.exp(-(ax**2) / (2*sigma**2)); g = g / g.sum()
    k = torch.outer(g, g)
    return k.view(1, 1, ksize, ksize).repeat(3, 1, 1, 1)   # depthwise, 3 ch

_GK = _gaussian_kernel()

def hf_energy_per_tile(tiles):
    # tiles:[B,3,h,w] in [0,1] -> [B] high-frequency energy score (one number per tile).
    k = _GK.to(tiles.device)
    blur = F.conv2d(tiles, k, padding=k.shape[-1]//2, groups=3)
    hf = (tiles - blur).abs().mean(dim=(1, 2, 3))
    return hf.cpu().numpy()

def clean_threshold(clean_hf_scores, k=3.0):
    # Robust baseline from clean tiles: median + k*MAD. Tiles above this are "noisy".
    med = np.median(clean_hf_scores)
    mad = np.median(np.abs(clean_hf_scores - med)) + 1e-8
    return med + k * 1.4826 * mad

def scan_tiles(image, thresh):
    # Break image into tiles, check each tile -> (per-tile score, per-tile noisy flag).
    hf = hf_energy_per_tile(tile_image(image))
    return hf, hf > thresh

def mask_tile_coverage(mask):
    # mask:[1,1,SIZE,SIZE] -> [GRID*GRID] fraction of each tile covered by the attack region.
    mt = tile_image(mask.repeat(1, 3, 1, 1))
    return mt.mean(dim=(1, 2, 3)).numpy()

def tile_metrics(true_ids, pred_ids):
    # Tile-level precision / recall / IoU (detected vs truly-attacked tiles).
    tp = int((true_ids & pred_ids).sum())
    fp = int((~true_ids & pred_ids).sum())
    fn = int((true_ids & ~pred_ids).sum())
    prec = tp / (tp + fp) if tp + fp else float("nan")
    rec  = tp / (tp + fn) if tp + fn else float("nan")
    union = int((true_ids | pred_ids).sum())
    iou = tp / union if union else float("nan")
    return dict(precision=prec, recall=rec, tile_IoU=iou, tp=tp, fp=fp, fn=fn)

## 8. Full per-image pipeline + the 4-panel comparison
For one image we: predict its class -> make an **irregular blob** region -> attack only that region
(FGSM or PGD) -> derive the **true attacked tiles** from blob coverage -> **scan the image tile by
tile** and flag noisy tiles -> grade with precision/recall/IoU. The figure shows, left to right:

1. **Original** image (clean).
2. **Adversarial** image with the **real attack region** drawn (yellow outline) over the tile grid —
   note the shape does not follow tile borders.
3. **Per-tile noise score** heat (what the detector measured on each tile).
4. **Detection**: truly-attacked tiles (lime, solid) vs **detected** tiles (cyan, dashed).

In [ ]:
# ============================================================
# CELL 8 - One-image pipeline + comparison figure
# ============================================================
def _outline_tiles(ax, ids, color, ls="-", lw=2.5):
    for k in np.where(ids)[0]:
        r, c = divmod(int(k), GRID)
        ax.add_patch(patches.Rectangle((c*TILE, r*TILE), TILE, TILE,
                     fill=False, edgecolor=color, linewidth=lw, linestyle=ls))

def _draw_grid(ax):
    for i in range(1, GRID):
        ax.axhline(i*TILE, color="white", lw=0.6, alpha=0.6)
        ax.axvline(i*TILE, color="white", lw=0.6, alpha=0.6)

def run_one(path_or_url, method="pgd", epsilon=0.05, k=3.0, seed=0,
            cover_thresh=0.05, full_image=False, show=True, name=""):
    x = load_image(path_or_url)                              # [1,3,SIZE,SIZE] CPU

    # 1) predict class, 2) build the irregular attack region, 3) attack only that region
    cls0, _ = batched_predict(x); cls0 = int(cls0[0])
    mask = None if full_image else random_blob_mask(seed=seed)
    x_adv = attack(x, cls0, method=method, epsilon=epsilon, mask=mask)

    # 4) ground truth: a tile is attacked if the blob covers > cover_thresh of it
    true_tiles = (np.ones(GRID*GRID, bool) if mask is None
                  else mask_tile_coverage(mask) > cover_thresh)

    # 5) calibrate on clean tiles, then scan the attacked image tile by tile
    thresh = clean_threshold(hf_energy_per_tile(tile_image(x)), k=k)
    hf_adv, flagged = scan_tiles(x_adv, thresh)
    m = tile_metrics(true_tiles, flagged)

    clsA, _ = batched_predict(x_adv); clsA = int(clsA[0])
    res = dict(name=name or os.path.basename(str(path_or_url)), method=method,
               before=idx2label[cls0], after=idx2label[clsA], fooled=clsA != cls0,
               n_true=int(true_tiles.sum()), n_flagged=int(flagged.sum()), **m)

    if show:
        orig_np, adv_np = to_np(x), to_np(x_adv)
        mask_np = mask.squeeze().numpy() if mask is not None else np.ones((SIZE, SIZE))
        fig, ax = plt.subplots(1, 4, figsize=(18, 4.8))
        ax[0].imshow(orig_np); ax[0].set_title("1) Original (clean)")
        ax[1].imshow(adv_np); _draw_grid(ax[1])
        ax[1].contour(mask_np, levels=[0.5], colors="yellow", linewidths=2.5)
        ax[1].set_title("2) Adversarial\nyellow = real attack region")
        ax[2].imshow(tile_grid_to_full(norm01(hf_adv)), cmap="hot")
        ax[2].set_title("3) Per-tile noise score")
        ax[3].imshow(adv_np)
        _outline_tiles(ax[3], true_tiles, "lime"); _outline_tiles(ax[3], flagged, "cyan", ls="--")
        ax[3].set_title("4) Detection\nlime = TRUE   cyan-- = DETECTED")
        for a in ax: a.axis("off")
        flag = "FOOLED" if res["fooled"] else "label held"
        fig.suptitle(f"{res['name']}  [{method.upper()}]  {res['before']} -> {res['after']} "
                     f"[{flag}]   tile-IoU={m['tile_IoU']:.2f}  "
                     f"P={m['precision']:.2f}  R={m['recall']:.2f}", y=1.04, fontsize=12)
        plt.tight_layout(); plt.show()
    return res

## 9. Run the whole workflow on the 10 images
Each image gets a different random blob region (`seed=i`) attacked with **PGD** (switch to
`method="fgsm"` for the faster/weaker attack). Panel 2 shows the irregular attack region over the
tile grid; panel 4 shows true vs detected tiles.

In [ ]:
# ============================================================
# CELL 9 - Run on all 10 images (irregular-region attack -> per-tile scan)
# ============================================================
ATTACK_METHOD = "pgd"     # "pgd" (stronger) or "fgsm" (faster)
EPSILON       = 0.05      # perturbation budget
K_SENSITIVITY = 3.0       # detector threshold: lower = more sensitive

results = [run_one(p, method=ATTACK_METHOD, epsilon=EPSILON, k=K_SENSITIVITY, seed=i, show=True)
           for i, p in enumerate(images)]

## 10. Summary table — how good is the detector?
Tile-level **precision / recall / IoU** per image (detected tiles vs the tiles the blob actually
touched), plus the averages and the global fool-rate. High recall = rarely misses an attacked tile;
high precision = rarely false-alarms a clean tile.

In [ ]:
# ============================================================
# CELL 10 - Summary table
# ============================================================
import pandas as pd
df = pd.DataFrame(results)
print(f"Attack: {ATTACK_METHOD.upper()}   epsilon={EPSILON}   k={K_SENSITIVITY}")
print(f"Global fool-rate: {df['fooled'].mean():.0%}")
print(f"Mean tile-IoU: {df['tile_IoU'].mean():.2f}   "
      f"Precision: {df['precision'].mean():.2f}   Recall: {df['recall'].mean():.2f}")
df[["name", "method", "before", "after", "fooled",
    "n_true", "n_flagged", "precision", "recall", "tile_IoU"]]

## Recap & knobs
- **Attack method:** `ATTACK_METHOD` (Cell 9) — `"fgsm"` (1 step, fast) or `"pgd"` (iterative,
  stronger, subtler noise). PGD params (`alpha`, `steps`) live in `pgd_attack`, Cell 5.
- **Attack region:** `random_blob_mask` (Cell 5) draws an irregular polygon; tweak `rad_frac` /
  `n_verts` for bigger or more jagged regions, or pass `full_image=True` to `run_one` to attack
  everything.
- **Attack strength:** `EPSILON` — raise it (`0.1`) for more visible/detectable noise, lower
  (`0.02`) to stress-test the detector.
- **Detector sensitivity:** `K_SENSITIVITY` / `k` — lower flags more tiles (higher recall, lower
  precision).
- **Ground-truth strictness:** `cover_thresh` in `run_one` — min blob coverage for a tile to count
  as truly attacked.
- **Granularity:** `GRID` (Cell 3) — finer grid localizes the region better but costs more tiles.

**Note on the baseline:** here the clean threshold is calibrated on each image's own clean tiles for
illustration. In a real deployment you'd calibrate once on a trusted clean set, then scan incoming
(possibly poisoned) images tile by tile with that fixed threshold.

**Next steps:** stronger/subtler attacks (more PGD steps, lower alpha) to test detector limits;
a finer `GRID` for tighter localization of the attacked region; or a learned per-tile classifier if
high-frequency energy proves too weak against low-frequency attacks.